# One Pizza

Ну эта задачка попроще предыдущей... у нас есть клиенты, каждый клиент что-то любит, что-то не любит, а что-то ему и вовсе безразлично. Надо придумать такой состав пиццы, чтобы к нам пришло как можно больше людей.   

ПОПРОЩЕ ЧЕМ ЭТИ ДАТАЦЕНТРЫ

Введем множества и индексы:  
$C$ - множество клиентов  
$L_c$ - множество ингридентов, которые клиенты любят  
$D_c$ - множество ингридиентов, которые клиенты не любят  
$I$ - множество всех ингридиентов (наименования грубо говоря)  

Хотим найти такое подмножество $P \subseteq I$, что $C_{involved}$ (число вовлеченных по итогу клиентов) $\rightarrow \max$, при условии $\begin{cases} L_{c_{involved}} \subseteq P \\ D_{c_{involved}} \cap P = \emptyset \end{cases}$  

# Переменные  

Для каждого ингридиента индикатор того, входит ли он в наше итоговое крутое подмножество  
**ИНДИКАТОР ДЛЯ ИНГРИДИЕНТА**  
$\forall i \ \in \ I: x_i = \begin{cases} 1, \ i \in P, \\ 0, \ else. \end{cases}$  
  
Для каждого клиента индикатор того, останется ли он доволен от нашего итогового рецепта  
**ИНДИКАТОР ДЛЯ КЛИЕНТА**  
$\forall c \ \in \ C: y_c = \begin{cases} 1, \ c \ доволен, \\ 0, \ else. \end{cases}$   

Соответственно $x_i \in \{0, 1\} \ , \ y_c \in \{0, 1\}$

# Ограничения  

 Мы считаем клиента довольным, если **все** его любимые ингридиенты в пицце.  
 $y_c \leq x_i \ \forall c, \ \forall i \ \in \ L_c$  

(соответственно если $\exists x_i = 0 \Longrightarrow y_c \leq 0 \Longrightarrow y_c = 0$)  

Мы считаем клиента довольным, если **ни один** из его **не** любимых ингридентов в пицце.  
$y_c \leq 1 - x_i \ \forall c , \forall i \ \in D$  

(соответственно если $\exists x_i = 1 \Longrightarrow y_c \leq 0 \Longrightarrow y_c = 0$)  

# Целевая функция  
$$\sum_{c=1}^{C} y_c \rightarrow \max$$

# Таким образом:  
$$\begin{cases} 
    \sum_{c=1}^{C}y_c \rightarrow \max \\
    y_c \leq x_i, \ \forall c \ \forall i \ \in L_c \\
    y_c \leq 1 - x_i, \ \forall c \ \forall i \in D_c \\
    x_i \in \{0, 1\}, \ \forall i \in I \\
    y_c \in \{0, 1\}, \ \forall c \in C \end{cases}$$

### Решение через SCIP

In [1]:
import os
import time
from ortools.linear_solver import pywraplp

In [2]:
INPUT_DIR = "labs/lab_3_2/input"
OUTPUT_DIR = "labs/lab_3_2/output"
TIME_LIMIT_MS = 60_000

In [3]:
def solve_file(filepath):
    with open(filepath, 'r') as f:
        data = f.read().strip().split()

        it = iter(data)
        C = int(next(it))

        likes = []
        dislikes = []
        all_ing = set()

        for _ in range(C):
            L = int(next(it))
            l = [next(it) for _ in range(L)]
            D = int(next(it))
            d = [next(it) for _ in range(D)]

            likes.append(l)
            dislikes.append(d)

            all_ing.update(l)
            all_ing.update(d)

        ing_list = sorted(all_ing)

        num_ing = len(ing_list)

        ing_to_idx = {name: i for i, name in enumerate(ing_list)}

        solver = pywraplp.Solver.CreateSolver("SCIP")

        solver.SetTimeLimit(TIME_LIMIT_MS)

        # переменные
        x = {i: solver.BoolVar(f"x_{i}") for i in range(num_ing)}
        y = {c: solver.BoolVar(f"y_{c}") for c in range(C)}

        # ограничения
        for c in range(C):
            for ing in likes[c]:
                solver.Add(y[c] <= x[ing_to_idx[ing]])
            for ing in dislikes[c]:
                solver.Add(y[c] <= 1 - x[ing_to_idx[ing]])

        objective = solver.Objective()
        for c in range(C):
            objective.SetCoefficient(y[c], 1)
        objective.SetMaximization()

        status = solver.Solve()

        if status == pywraplp.Solver.OPTIMAL:
            status_str = "OPTIMAL"
        elif status == pywraplp.Solver.FEASIBLE:
            status_str = "FEASIBLE"
        else: 
            return None, 0, f"FAILED({status})", C, num_ing

        chosen = [ing_list[i] for i in range(num_ing) if x[i].solution_value() > 0.5]
        # не хотим проблем с округлением, поэтому пишем x > 0.5  

        score = round(objective.Value())
        return chosen, score, status_str, C, num_ing


In [4]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for fname in sorted(os.listdir(INPUT_DIR)):

        in_path = os.path.join(INPUT_DIR, fname)
        out_path = os.path.join(OUTPUT_DIR, fname + '_SCIP_' + '.out')

        print(f"FILE: {fname}")

        t0 = time.time()
        chosen, score, status_str, C, num_ing = solve_file(in_path)
        elapsed = time.time() - t0

        if chosen is None:
            print(f"клиентов: {C}, ингредиентов: {num_ing}")
            print(f"статус: {status_str}")
            print("Решение не найдено.")
            continue

        with open(out_path, 'w') as f:
            if chosen:
                f.write(f"{len(chosen)} " + " ".join(chosen) + "\n")
            else:
                f.write("0\n")

        print(f"клиентов: {C}, ингредиентов: {num_ing}")
        print(f"status: {status_str}")
        print(f"score: {score}")
        print(f"время: {elapsed:.1f} сек")
        print()


if __name__ == "__main__":
    main()

FILE: a_an_example.in.txt
клиентов: 3, ингредиентов: 6
status: OPTIMAL
score: 2
время: 0.0 сек

FILE: b_basic.in.txt
клиентов: 5, ингредиентов: 6
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: c_coarse.in.txt
клиентов: 10, ингредиентов: 10
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: d_difficult.in.txt
клиентов: 9368, ингредиентов: 600
status: FEASIBLE
score: 1707
время: 59.6 сек

FILE: e_elaborate.in.txt
клиентов: 4986, ингредиентов: 10000
status: FEASIBLE
score: 1676
время: 60.1 сек



FILE: a_an_example.in.txt
клиентов: 3, ингредиентов: 6
status: OPTIMAL
score: 2
время: 0.0 сек

FILE: b_basic.in.txt
клиентов: 5, ингредиентов: 6
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: c_coarse.in.txt
клиентов: 10, ингредиентов: 10
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: d_difficult.in.txt
клиентов: 9368, ингредиентов: 600
status: FEASIBLE
score: 1707
время: 59.6 сек

FILE: e_elaborate.in.txt
клиентов: 4986, ингредиентов: 10000
status: FEASIBLE
score: 1676
время: 60.1 сек

### CP-SAT вроде как должен быть побыстрее

In [10]:
from ortools.sat.python import cp_model

TIME_LIMIT_S = 150

In [11]:
def solve_file(filepath):
    with open(filepath, 'r') as f:
        data = f.read().strip().split()

        it = iter(data)
        C = int(next(it))

        likes = []
        dislikes = []
        all_ing = set()

        for _ in range(C):
            L = int(next(it))
            l = [next(it) for _ in range(L)]
            D = int(next(it))
            d = [next(it) for _ in range(D)]

            likes.append(l)
            dislikes.append(d)

            all_ing.update(l)
            all_ing.update(d)

        ing_list = sorted(all_ing)

        num_ing = len(ing_list)

        ing_to_idx = {name: i for i, name in enumerate(ing_list)}

    model = cp_model.CpModel()

    x = {i: model.NewBoolVar(f'x_{i}') for i in range(num_ing)}
    y = {c: model.NewBoolVar(f'y_{c}') for c in range(C)}

    for c in range(C):
        for ing in likes[c]:
            # y_c => x_i   (если клиент доволен, любимый ингредиент обязан быть)
            model.AddImplication(y[c], x[ing_to_idx[ing]])
        for ing in dislikes[c]:
            # y_c => not x_i
            model.AddImplication(y[c], x[ing_to_idx[ing]].Not())

    model.Maximize(sum(y.values()))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = TIME_LIMIT_S
    solver.parameters.num_search_workers = 8

    status = solver.Solve(model)

    if status == cp_model.OPTIMAL:
        status_str = "OPTIMAL"
    elif status == cp_model.FEASIBLE:
        status_str = "FEASIBLE"
    elif status == cp_model.INFEASIBLE:
        status_str = "INFEASIBLE"
    else:
        status_str = "UNKNOWN"


    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None, 0, status_str, C, num_ing

    chosen = [ing_list[i] for i in range(num_ing) if solver.Value(x[i]) == 1]
    score = round(solver.ObjectiveValue())
    return chosen, score, status_str, C, num_ing

In [12]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for fname in sorted(os.listdir(INPUT_DIR)):

        in_path = os.path.join(INPUT_DIR, fname)
        out_path = os.path.join(OUTPUT_DIR, fname + '_CP-SAT_' + '.out')

        print(f"FILE: {fname}")

        t0 = time.time()
        chosen, score, status_str, C, num_ing = solve_file(in_path)
        elapsed = time.time() - t0

        if chosen is None:
            print(f"клиентов: {C}, ингредиентов: {num_ing}")
            print(f"статус: {status_str}")
            print("Решение не найдено.")
            continue

        with open(out_path, 'w') as f:
            if chosen:
                f.write(f"{len(chosen)} " + " ".join(chosen) + "\n")
            else:
                f.write("0\n")

        print(f"клиентов: {C}, ингредиентов: {num_ing}")
        print(f"status: {status_str}")
        print(f"score: {score}")
        print(f"время: {elapsed:.1f} сек")
        print()


if __name__ == "__main__":
    main()

FILE: a_an_example.in.txt
клиентов: 3, ингредиентов: 6
status: OPTIMAL
score: 2
время: 0.0 сек

FILE: b_basic.in.txt
клиентов: 5, ингредиентов: 6
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: c_coarse.in.txt
клиентов: 10, ингредиентов: 10
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: d_difficult.in.txt
клиентов: 9368, ингредиентов: 600
status: OPTIMAL
score: 1805
время: 6.7 сек

FILE: e_elaborate.in.txt
клиентов: 4986, ингредиентов: 10000
status: FEASIBLE
score: 2080
время: 150.6 сек



FILE: a_an_example.in.txt
клиентов: 3, ингредиентов: 6
status: OPTIMAL
score: 2
время: 0.0 сек

FILE: b_basic.in.txt
клиентов: 5, ингредиентов: 6
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: c_coarse.in.txt
клиентов: 10, ингредиентов: 10
status: OPTIMAL
score: 5
время: 0.0 сек

FILE: d_difficult.in.txt
клиентов: 9368, ингредиентов: 600
status: OPTIMAL
score: 1805
время: 6.7 сек

FILE: e_elaborate.in.txt
клиентов: 4986, ингредиентов: 10000
status: FEASIBLE
score: 2080
время: 150.6 сек